<br><h1>Fake News Detection Part 2</h1><br>

In [1]:
# libraries for part 1
import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.model_selection import train_test_split
import re
import sys

pd.set_option('display.max_columns', 200)
plt.style.use("ggplot")

# Libraries for part 2
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import scipy.sparse
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

import gc
from IPython.display import clear_output

### Task 1 - grouping the labels
To group the labels in two groups, we first identified the unique labels used. This also guarantees that the drop before went as wished.

In [2]:
df = pl.read_csv("./data/995,000_rows_preprocessed.csv").to_pandas()
print(df['type'].value_counts())

del df
gc.collect(); # remove big dataframe to save RAM

type
reliable                      218564
political                     194518
bias                          133232
fake                          104883
conspiracy                     97314
rumor                          56445
nan                            47786
unknown                        43534
unreliable                     35332
clickbait                      27412
junksci                        14040
satire                         13160
hate                            8779
2018-02-10 13:43:39.521661         1
Name: count, dtype: int64


In [2]:
df = pl.read_csv("./data/995,000_rows_preprocessed.csv").to_pandas()
df = df[['type', 'content']].rename(columns={'type':'isfake'}) # rename col for binary

real_labels = ['reliable', 'political']
fake_labels = ['bias','fake','unreliable','rumor','conspiracy','clickbait','junksci','satire']
drop_labels = ['unknown', 'hate', 'nan', '2018-02-10 13:43:39.521661']

df = df.loc[~df['isfake'].isin(drop_labels)]
df['isfake'] = df['isfake'].map(lambda x : 0 if x in real_labels else 1)

n = df.shape[0]
indices = np.arange(n)
labels = df['isfake'].to_numpy()

# First split: 80 % train, 20 % temp (will become val + test)
train_idx, temp_idx = train_test_split(indices, test_size=0.2, random_state=1000, stratify=labels)

# Second split: halve the temp set into 10 % val + 10 % test
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=1000, stratify=labels[temp_idx])

df_train = df.iloc[train_idx]
df_val = df.iloc[val_idx]
df_test = df.iloc[test_idx]

print(df_train.shape)
print(df_val.shape)
print(df_test.shape)

del df
gc.collect();

(715920, 2)
(89490, 2)
(89490, 2)


### Task 2 - logistic regression
To train the logistic classifier, we started by converting the contents to 10,000 frequency-based columns based on the top words.

**Bag of Words function:** Given a pandas or polars dataframe and potentially an number of tokens  to be included as features, returns a bag of words representation of the input's column 'content' as a scipy.sparse.csr_matrix.

In [3]:
def bow(df, vectorizer, col, fit_transform=True, num_feats=10**4, save_as=None):
    # Define corpus using polars for efficiency
    if type(df) == pd.DataFrame: pl_series = pl.from_pandas(df[col])
    else: pl_series = df.select(col)

    # Concat natural language cols to single list
    docs = pl_series.to_list()

    # Fit vectorizer to create compressed sparse column matrix for RAM efficiency
    if fit_transform: X = vectorizer.fit_transform(docs)
    else: X = vectorizer.transform(docs)

    # save as numpyzip
    if save_as: scipy.sparse.save_npz(save_as, X)
    return X

In [1]:
# # Create frequency-based representation of training, validation and test set

# # Initialize vectorizer
vectorizers = [
    CountVectorizer(max_features=int(1e4), analyzer='word', token_pattern=r'\w{1,}', ngram_range=(1,3), binary=True), # Binary BOW
    CountVectorizer(max_features=int(1e4), analyzer='word', token_pattern=r'\w{1,}', ngram_range=(1,3)) # Count BOW
]

for vec in vectorizers:
    print(vec)

    X_train = bow(df_train, vec, 'content', fit_transform=True)
    X_val = bow(df_val, vec, 'content', fit_transform=False)
    X_test = bow(df_test, vec, 'content', fit_transform=False)
    
    y_train = df_train['isfake']
    y_val = df_val['isfake']
    y_test = df_test['isfake']
    
    # smote oversampling
    smoter = SMOTE(sampling_strategy='minority', random_state=1000)
    X_train_sm, y_train_sm = smoter.fit_resample(X_train, y_train)
    
    classifier = LogisticRegression(C=1.0, max_iter=1000, random_state=1000)
    classifier.fit(X_train_sm, y_train_sm)
    preds = classifier.predict(X_val)
    print(classification_report(preds, y_val), end='\n\n' + '='*60 + '\n\n\n')

NameError: name 'CountVectorizer' is not defined

In [ ]:
# test set
vec = vectorizers[0]

X_train = bow(df_train, vec, 'content', fit_transform=True)
X_test = bow(df_test, vec, 'content', fit_transform=False)
X_train_sm, y_train_sm = smoter.fit_resample(X_train, y_train)

classifier = LogisticRegression(C=1.0, max_iter=1000, random_state=1000)
classifier.fit(X_train_sm, y_train_sm)
preds = classifier.predict(X_test)
print('accuracy: {}'.format(accuracy_score(preds, y_test)))
print('f1-score: {}'.format(f1_score(preds, y_test)))
print(classification_report(preds, y_test))

### Task 3 - Inclusion of metadata
We start by reading in the data again. Then we treat it as a kaggle problem, starting with most or all columns and strategically excluding the ones that make the loss go up.

In [11]:
df = pl.read_csv("./data/995,000_rows_preprocessed.csv").to_pandas()
df = df.rename(columns={'type':'isfake'}) # rename col for binary

# drop columns we deem mostly useless
df.drop(['url','scraped_at','inserted_at','updated_at'], axis=1, inplace=True)

real_labels = ['reliable', 'political']
fake_labels = ['bias','fake','unreliable','rumor','conspiracy','clickbait','junksci','satire']
drop_labels = ['unknown', 'hate', 'nan', '2018-02-10 13:43:39.521661']

df = df.loc[~df['isfake'].isin(drop_labels)].iloc[:20000] # limit to 20000 samples
df['isfake'] = df['isfake'].map(lambda x : 0 if x in real_labels else 1)

for col in df:
    print(f'Col: {col}, #Unique: {len(df[col].unique())}, nan: {(df[col] == "nan").sum() / n:.3f}')

Col: domain, #Unique: 381, nan: 0.000
Col: isfake, #Unique: 2, nan: 0.000
Col: content, #Unique: 17412, nan: 0.000
Col: title, #Unique: 17918, nan: 0.009
Col: authors, #Unique: 5160, nan: 0.424
Col: meta_keywords, #Unique: 5636, nan: 0.650
Col: meta_description, #Unique: 8338, nan: 0.557
Col: tags, #Unique: 2942, nan: 0.819
Col: source, #Unique: 3, nan: 0.766


Now we'll convert some of the most popular domains and authors to categorical features for prediction.

In [12]:
# categorical values
cat_cols = ['domain','authors','source']

# 100 cols max
for col in cat_cols:
    top_vals = df[col].value_counts().index[:99]
    df[col] = df[col].apply(lambda x : x if x in top_vals else '<OTHER>')

# aggregate string columns for approximations
agg_cols = ['title','meta_keywords','meta_description','tags']
df['meta_str_agg'] = df[agg_cols].sum(axis=1)
df.drop(agg_cols, axis=1, inplace=True)

for col in df:
    print(f'Col: {col}, #Unique: {len(df[col].unique())}, nan: {(df[col] == "nan").sum() / n:.3f}')

cat_cols = ['domain','authors','source']
cat_dummies = pd.get_dummies(df[cat_cols], dtype='int')
df = pd.concat([df.drop(cat_cols, axis=1), cat_dummies], axis=1)

Col: domain, #Unique: 100, nan: 0.000
Col: isfake, #Unique: 2, nan: 0.000
Col: content, #Unique: 17412, nan: 0.000
Col: authors, #Unique: 100, nan: 0.424
Col: source, #Unique: 3, nan: 0.766
Col: meta_str_agg, #Unique: 18514, nan: 0.000


In [13]:
# train-val-test split
n = df.shape[0]
indices = np.arange(n)
labels = df['isfake'].to_numpy()

# First split: 80 % train, 20 % temp (will become val + test)
train_idx, temp_idx = train_test_split(indices, test_size=0.2, random_state=1000, stratify=labels)

# Second split: halve the temp set into 10 % val + 10 % test
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=1000, stratify=labels[temp_idx])

df_train = df.iloc[train_idx]
df_val = df.iloc[val_idx]
df_test = df.iloc[test_idx]

print(df_train.shape)
print(df_val.shape)
print(df_test.shape)

(16000, 206)
(2000, 206)
(2000, 206)


In [14]:
# Create frequency-based representation of training, validation and test set

# Initialize vectorizer
vectorizers = [
    TfidfVectorizer(max_features=int(1e4), ngram_range=(1,3), sublinear_tf=False), # TFIDF
    TfidfVectorizer(max_features=int(1e4), ngram_range=(1,3), sublinear_tf=True), # Log TFIDF
    CountVectorizer(max_features=int(1e4), ngram_range=(1,3), binary=True), # Binary BOW
    CountVectorizer(max_features=int(1e4), ngram_range=(1,3), binary=False) # Count BOW
]

for vec in vectorizers:
    print(vec)

    X_train = bow(df_train, vec, 'content', fit_transform=True)
    X_val = bow(df_val, vec, 'content', fit_transform=False)
    X_test = bow(df_test, vec, 'content', fit_transform=False)
    X_train = bow(df_train, vec, 'meta_str_agg', fit_transform=True)
    X_val = bow(df_val, vec, 'meta_str_agg', fit_transform=False)
    X_test = bow(df_test, vec, 'meta_str_agg', fit_transform=False)
    
    y_train = df_train['isfake']
    y_val = df_val['isfake']
    y_test = df_test['isfake']
    
    # smote oversampling
    smoter = SMOTE(sampling_strategy='minority', random_state=1000)
    X_train_sm, y_train_sm = smoter.fit_resample(X_train, y_train)
    
    classifier = LogisticRegression(max_iter=1000, random_state=1000)
    classifier.fit(X_train_sm, y_train_sm)
    preds = classifier.predict(X_val)
    print(classification_report(preds, y_val), end='\n\n' + '='*60 + '\n\n\n')

TfidfVectorizer(max_features=10000, ngram_range=(1, 3))
              precision    recall  f1-score   support

           0       0.79      0.78      0.78       879
           1       0.83      0.83      0.83      1121

    accuracy                           0.81      2000
   macro avg       0.81      0.81      0.81      2000
weighted avg       0.81      0.81      0.81      2000




TfidfVectorizer(max_features=10000, ngram_range=(1, 3), sublinear_tf=True)
              precision    recall  f1-score   support

           0       0.77      0.77      0.77       870
           1       0.83      0.83      0.83      1130

    accuracy                           0.80      2000
   macro avg       0.80      0.80      0.80      2000
weighted avg       0.80      0.80      0.80      2000




CountVectorizer(binary=True, max_features=10000, ngram_range=(1, 3))
              precision    recall  f1-score   support

           0       0.79      0.74      0.76       932
           1       0.78      0.

In [16]:
# test set
vec = vectorizers[0]

X_train = bow(df_train, vec, 'content', fit_transform=True)
X_test = bow(df_test, vec, 'content', fit_transform=False)
X_train = bow(df_train, vec, 'meta_str_agg', fit_transform=True)
X_test = bow(df_test, vec, 'meta_str_agg', fit_transform=False)
X_train_sm, y_train_sm = smoter.fit_resample(X_train, y_train)

classifier = LogisticRegression(max_iter=1000, random_state=1000)
classifier.fit(X_train_sm, y_train_sm)
preds = classifier.predict(X_test)
print(classification_report(preds, y_test))

              precision    recall  f1-score   support

           0       0.78      0.79      0.78       857
           1       0.84      0.83      0.83      1143

    accuracy                           0.81      2000
   macro avg       0.81      0.81      0.81      2000
weighted avg       0.81      0.81      0.81      2000

